# 3-8. Adaptive RAG — 질문 복잡도에 따른 검색 경로 분기

🧩 심화/선택

<!-- optional -->

## 학습 목표
- 쿼리의 성격에 따라 검색 경로를 **No-retrieval / Single-step / Multi-step** 세 갈래로 분기하는 adaptive RAG를 설계할 수 있다.
- Pydantic schema + LLM 구조화 출력으로 **쿼리 라우터**를 구현할 수 있다.
- LangGraph `StateGraph`의 `add_conditional_edges`로 라우팅 로직을 명시적 그래프로 표현할 수 있다.
- Agentic RAG(매 턴 결정)와 Adaptive RAG(초기 1회 결정)의 **운영·감사 관점 차이**를 비교할 수 있다.

## 핵심 키워드
`adaptive RAG` · `query router` · `query complexity classifier` · `conditional edge (LangGraph)` · `multi-hop retrieval`

> 🧩 이 노트북은 **심화/선택** 과정이다. S3-5(Self-RAG)와 S3-7(Agentic RAG)를 경험한 뒤 읽으면 세 기법의 결정 주체·비용·감사성이 명확히 대비된다.

## 1. Adaptive RAG 개념 (Jeong et al., 2024)

[*Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity*](https://arxiv.org/abs/2403.14403)는 질문을 세 가지 복잡도로 분류해 서로 다른 파이프라인을 태운다는 단순하지만 강력한 아이디어다.

| 복잡도 | 예시 | 파이프라인 |
|--------|------|----------|
| **A. No-retrieval** | "안녕하세요", "당신은 누구" | LLM만 사용 (검색 없음) |
| **B. Single-step** | "청약철회 기간은?" | retrieval 1회 + 생성 |
| **C. Multi-step** | "A사 자회사가 발행한 상품의 약관상 철회 기간은?" | 다단계 검색 + 중간 합성 |

- **검색 비용을 절약**: 쉬운 질문에 retrieval 돌릴 필요가 없다.
- **복잡 질문의 정확도 향상**: 한 번의 검색으로 부족한 정보를 여러 번 나눠 조회한다.
- **경로가 사전 결정**: Agentic RAG와 달리 **초반 1회 라우팅 후 경로가 고정**되므로 감사 로그가 읽기 쉽다.

```mermaid
flowchart TD
    Q[질문] --> R{classify_query}
    R -- no_retrieval --> D[direct_answer]
    R -- single_retrieval --> S[single_rag]
    R -- multi_retrieval --> M[multi_hop_rag]
    D --> G[generate]
    S --> G
    M --> G
    G --> END([최종 답변])
```

In [ ]:
# 공통 세팅
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

## 2. 약관 벡터스토어 준비 (S2-4와 동일)

캡스톤 5개 질문과 추가 2개 질문으로 라우터가 올바른 경로를 선택하는지 평가한다. 코퍼스는 단독 재현을 위해 노트북 내부에 포함했다.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

E_FINANCE_TERMS = [
    Document(page_content='제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함으로써 이용자의 권익 보호와 거래의 안전을 확보함을 목적으로 한다.', metadata={'article': '제1조'}),
    Document(page_content='제2조(정의) 전자금융거래란 금융회사 또는 전자금융업자가 전자적 장치를 통해 제공하는 금융상품 및 서비스를 자동화된 방식으로 이용하는 거래를 말한다.', metadata={'article': '제2조'}),
    Document(page_content='제5조(청약철회) ① 이용자는 계약 체결 후 14일 이내에 서면 또는 전자문서로 청약을 철회할 수 있다.', metadata={'article': '제5조'}),
    Document(page_content='제8조(수수료) 전자금융거래 이용 수수료는 거래 유형별로 사전에 안내되며, 수수료 인상 시 적용일 30일 전에 공지해야 한다.', metadata={'article': '제8조'}),
    Document(page_content='제10조(분실신고) ① 이용자는 접근매체의 분실·도난 시 즉시 고객센터 또는 가장 가까운 영업점에 신고하여야 한다.', metadata={'article': '제10조'}),
    Document(page_content='제14조(이의신청) ① 이용자는 거래 내역에 이의가 있을 때 거래일로부터 1년 이내에 이의를 신청할 수 있다. ② 금융회사는 이의신청 접수일로부터 15영업일 이내에 결과를 통지한다.', metadata={'article': '제14조'}),
    Document(page_content='제17조(서비스 이용시간) 전자금융거래 서비스는 24시간 이용을 원칙으로 하되, 시스템 점검·장애·운영상 필요에 따라 일시적으로 제한될 수 있다.', metadata={'article': '제17조'}),
    # Multi-hop 시나리오를 위한 보조 문서
    Document(page_content='삼성전자의 자회사로는 삼성카드와 삼성생명이 있으며, 이들은 각각 신용카드 및 보험 상품을 발행한다.', metadata={'article': '그룹구조'}),
    Document(page_content='삼성카드가 발행하는 신용카드 상품은 전자금융거래 표준약관을 준용하며, 청약철회는 제5조에 따라 14일 이내에 가능하다.', metadata={'article': '상품안내'}),
]

vs = Chroma.from_documents(
    E_FINANCE_TERMS,
    embedding=get_embeddings(),
    collection_name='efinance_adaptive',
)
retriever = vs.as_retriever(search_kwargs={'k': 3})
llm = get_chat_model(temperature=0)
print(f'약관 {len(E_FINANCE_TERMS)}개 인덱싱 완료')

## 3. 쿼리 복잡도 분류기 (라우터)

LLM에게 `route` 필드를 가진 Pydantic 객체를 생성하게 해 **구조화 출력**을 얻는다. 파싱 실패가 없고 다운스트림 `add_conditional_edges`에 그대로 사용할 수 있다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class RouteDecision(BaseModel):
    """질문을 세 가지 경로 중 하나로 분류한다."""
    route: Literal['no_retrieval', 'single_retrieval', 'multi_retrieval'] = Field(
        description=(
            'no_retrieval: 일상 대화/인사/LLM 일반 지식으로 충분. '
            'single_retrieval: 약관 1회 조회로 답할 수 있는 단일 사실 질의. '
            'multi_retrieval: 2개 이상의 문서를 연결해야 답할 수 있는 다단계 질의.'
        )
    )
    reasoning: str = Field(description='1문장으로 분류 근거를 밝힌다.')

router_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 금융 도메인 RAG 시스템의 쿼리 라우터입니다.\n'
     '질문을 보고 세 경로 중 하나로 분류하세요.\n'
     '- 일상 대화(인사, 사소한 잡담)는 no_retrieval.\n'
     '- 약관 한 조항만 보면 답할 수 있으면 single_retrieval.\n'
     '- 엔티티 연결(예: 자회사→상품→약관)이 필요하면 multi_retrieval.'),
    ('user', '{question}'),
])

router = router_prompt | llm.with_structured_output(RouteDecision)

# 동작 확인
for q in ['안녕', '청약철회 기간은?', '삼성전자 자회사가 발행한 상품의 철회 기간은?']:
    d = router.invoke({'question': q})
    print(f'{q!r:60s} → {d.route}  ({d.reasoning})')

## 4. LangGraph StateGraph — 조건부 분기

상태에 `route`, `documents`, `generation`을 두고, `classify_query` 노드가 라우팅한 결과에 따라 세 갈래로 나뉜 뒤 `generate`로 수렴한다.

In [ ]:
from typing import TypedDict
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

class AdaptiveState(TypedDict):
    question: str
    route: str
    documents: list[Document]
    generation: str
    trace: list[str]   # 경유한 노드 이름을 누적


def classify_query(state: AdaptiveState) -> AdaptiveState:
    d = router.invoke({'question': state['question']})
    state['route'] = d.route
    state['trace'] = state.get('trace', []) + [f'classify_query→{d.route}']
    return state


def direct_answer(state: AdaptiveState) -> AdaptiveState:
    # 검색 없이 LLM 일반 지식으로만 응답 — 약관은 건드리지 않는다
    state['documents'] = []
    state['trace'].append('direct_answer')
    return state


def single_rag(state: AdaptiveState) -> AdaptiveState:
    state['documents'] = retriever.invoke(state['question'])
    state['trace'].append(f'single_rag(k={len(state["documents"])})')
    return state


# multi-hop: 1차 검색 → LLM이 후속 서브쿼리 1개 생성 → 2차 검색 → 합성
hop_prompt = ChatPromptTemplate.from_template(
    '원 질문: {q}\n'
    '1차 검색 결과:\n{ctx}\n\n'
    '위 결과에서 아직 답하지 못한 정보를 얻기 위한 **후속 검색어 하나**만 출력하세요. '
    '한 줄만 출력하고, 따옴표/설명 없이 검색어만 쓰세요.'
)
hop_chain = hop_prompt | llm | StrOutputParser()

def multi_hop_rag(state: AdaptiveState) -> AdaptiveState:
    # Hop 1
    docs1 = retriever.invoke(state['question'])
    ctx1 = '\n'.join(d.page_content for d in docs1)
    # Hop 2 — LLM이 생성한 서브쿼리로 추가 검색
    followup = hop_chain.invoke({'q': state['question'], 'ctx': ctx1}).strip()
    docs2 = retriever.invoke(followup)
    # 중복 제거
    seen, merged = set(), []
    for d in docs1 + docs2:
        key = d.page_content[:40]
        if key not in seen:
            seen.add(key)
            merged.append(d)
    state['documents'] = merged
    state['trace'].append(f'multi_hop_rag(hop1={len(docs1)}, followup={followup!r}, total={len(merged)})')
    return state


gen_prompt = ChatPromptTemplate.from_template(
    '아래 컨텍스트를 참고해 한국어로 간결히 답하세요. '
    '컨텍스트가 비어 있으면 일반 지식으로 답하되 "(근거 없음)" 문구를 끝에 붙이세요.\n\n'
    '[컨텍스트]\n{ctx}\n\n[질문] {q}\n[답변]'
)
gen_chain = gen_prompt | llm | StrOutputParser()

def generate(state: AdaptiveState) -> AdaptiveState:
    ctx = '\n---\n'.join(d.page_content for d in state['documents']) if state['documents'] else ''
    state['generation'] = gen_chain.invoke({'ctx': ctx, 'q': state['question']})
    state['trace'].append('generate')
    return state


def route_selector(state: AdaptiveState) -> str:
    return state['route']

In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(AdaptiveState)
g.add_node('classify', classify_query)
g.add_node('direct', direct_answer)
g.add_node('single', single_rag)
g.add_node('multi', multi_hop_rag)
g.add_node('generate', generate)

g.set_entry_point('classify')
g.add_conditional_edges('classify', route_selector, {
    'no_retrieval': 'direct',
    'single_retrieval': 'single',
    'multi_retrieval': 'multi',
})
# 세 갈래 모두 generate로 수렴
for node in ('direct', 'single', 'multi'):
    g.add_edge(node, 'generate')
g.add_edge('generate', END)

adaptive_app = g.compile()
print('Adaptive RAG 그래프 컴파일 완료')

## 5. 벤치마크 — 캡스톤 5개 + 추가 2개로 경로 선택 검증

S2-4에서 생성된 `_capstone_baseline.json`이 있으면 불러오고, 없으면 S2-4와 동일한 5개 질문을 인라인으로 사용한다. 여기에 다음 2개를 더한다.

- 일상 대화: **"오늘 날씨 어때"** → `no_retrieval` 기대
- 다단계: **"삼성전자의 자회사가 발행한 상품의 약관에 따른 청약철회 기간은?"** → `multi_retrieval` 기대

In [ ]:
import json
from pathlib import Path

# 캡스톤 베이스라인 로드 (없으면 fallback)
BASELINE_PATH = Path('../../day1/session2_rag_basics/_capstone_baseline.json')
if BASELINE_PATH.exists():
    capstone_qs = [row['question'] for row in json.loads(BASELINE_PATH.read_text(encoding='utf-8'))]
    print(f'✅ S2-4 베이스라인에서 {len(capstone_qs)}개 질문 로드')
else:
    capstone_qs = [
        '청약철회 기간은 며칠인가?',
        '카드 등 접근매체 분실 시 신고 절차는 어떻게 되나?',
        '수수료를 인상하려면 며칠 전에 공지해야 하는가?',
        '거래 내역에 이의가 있을 때 언제까지 이의신청을 할 수 있고, 금융회사의 회신 기한은 얼마인가?',
        '서비스 이용시간은 어떻게 되며, 일시 제한 사유에는 무엇이 있는가?',
    ]
    print(f'⚠️ 베이스라인 없음 — fallback {len(capstone_qs)}개 사용 (S2-4 먼저 실행 권장)')

# 추가 2개
extra_qs = [
    ('오늘 날씨 어때', 'no_retrieval'),
    ('삼성전자의 자회사가 발행한 상품의 약관에 따른 청약철회 기간은?', 'multi_retrieval'),
]

all_qs = [(q, 'single_retrieval') for q in capstone_qs] + extra_qs
print(f'총 평가 대상: {len(all_qs)}개')

In [ ]:
import pandas as pd

rows = []
for q, expected in all_qs:
    init: AdaptiveState = {
        'question': q,
        'route': '',
        'documents': [],
        'generation': '',
        'trace': [],
    }
    final = adaptive_app.invoke(init)
    rows.append({
        '질문': q if len(q) < 60 else q[:57] + '…',
        '기대 경로': expected,
        '실제 경로': final['route'],
        '일치': '✅' if final['route'] == expected else '❌',
        '검색 문서 수': len(final['documents']),
        '답변 길이': len(final['generation']),
    })

df = pd.DataFrame(rows)
df

In [ ]:
# 특정 질문의 상세 trace와 답변을 확인
sample_q = '삼성전자의 자회사가 발행한 상품의 약관에 따른 청약철회 기간은?'
sample = adaptive_app.invoke({
    'question': sample_q,
    'route': '',
    'documents': [],
    'generation': '',
    'trace': [],
})

print('=== TRACE ===')
for step in sample['trace']:
    print(' -', step)
print(f'\n=== 검색된 문서 ({len(sample["documents"])}개) ===')
for d in sample['documents']:
    print(f'  [{d.metadata.get("article")}] {d.page_content[:70]}...')
print(f'\n=== 최종 답변 ===\n{sample["generation"]}')

## 6. Agentic RAG vs Adaptive RAG 비교

| 관점 | Agentic RAG (S3-7) | Adaptive RAG (S3-8) |
|------|--------------------|---------------------|
| **결정 주체** | LLM이 매 턴 tool_call 여부를 결정 | 라우터가 **초기 1회** 경로 결정 후 고정 |
| **제어 흐름** | 루프 (agent↔tools) | 분기 (classify→1 of N→generate) |
| **도구 유연성** | 임의의 tool 추가 가능, 조합 자유 | 경로별 파이프라인은 사전 정의 |
| **감사 용이성** | 매 호출마다 다른 trace → 로그 해석 어려움 | 경로가 유한·예측 가능 → 감사 친화적 |
| **비용 예측** | 상한만 설정 가능(`recursion_limit`) | 경로별 상한이 설계 시점에 고정 |
| **적합 도메인** | 복잡·개방형 지식 탐색, 내부 자동화 | 규제/금융처럼 **설명 가능성이 중요한** 도메인 |
| **실패 모드** | 도구 호출 폭주, 잘못된 tool 선택 | 라우터 오분류 → 잘못된 경로로 고정 |

> 🏦 **실무 팁**: 금융권 초기 배포에는 **Adaptive RAG**로 감사성을 확보하고, 내부 툴(장애 분석·로그 QA) 에이전트에는 **Agentic RAG**를 쓰는 조합이 현실적이다.

## 7. Session 3 종합 의사결정 트리 — "언제 어느 기법을 쓰나"

S3-1 ~ S3-8을 모두 마친 시점에, 새로운 RAG 요구사항을 받으면 아래 순서로 판단하라.

| # | 제목 | 핵심 | 쓰는 시점 |
|---|------|------|----------|
| 3-1 | Reranking | 2-stage (dense→cross-encoder) | **언제나 기본 탑재** — top-k 정확도가 성능의 80%를 좌우 |
| 3-2 | Hybrid Ensemble | BM25 + dense + RRF | 정확한 용어 질의와 의미 질의가 섞일 때 |
| 3-3 | Query Transform | HyDE, multi-query | 질문이 짧거나 모호해 recall이 낮을 때 |
| 3-4 | Contextual Compression | LLMChainExtractor, Embeddings filter | 긴 청크를 LLM에 주기 전에 노이즈 제거 |
| 3-5 | Self-RAG / CRAG | 자가교정 루프 | 검색 품질이 불안정하고 재시도가 필요할 때 |
| 3-6 | GraphRAG | Neo4j + MS GraphRAG | 엔티티 관계·다단계 사실 연결이 본질인 질의 |
| 3-7 | Agentic RAG 🧩 | LLM이 tool을 직접 호출 | 내부 자동화, 계산·검색·API가 섞인 복합 태스크 |
| 3-8 | Adaptive RAG 🧩 | 라우터 분기 | 감사·비용 예측이 중요한 **운영 배포**의 진입점 |

```mermaid
flowchart TD
    START[새 RAG 요구사항] --> Q1{질문 유형이 다양한가?}
    Q1 -- 예 --> Q2{감사/규제 요건이 강한가?}
    Q2 -- 예 --> ADAPTIVE[3-8 Adaptive RAG]
    Q2 -- 아니오 --> AGENTIC[3-7 Agentic RAG]
    Q1 -- 아니오 --> Q3{엔티티 관계가 핵심인가?}
    Q3 -- 예 --> GRAPH[3-6 GraphRAG]
    Q3 -- 아니오 --> Q4{검색 품질이 불안정한가?}
    Q4 -- 예 --> SELF[3-5 Self/CRAG]
    Q4 -- 아니오 --> BASE[3-1 Rerank + 3-2 Hybrid]
    ADAPTIVE -.항상 포함.-> BASE
    AGENTIC -.항상 포함.-> BASE
    GRAPH -.항상 포함.-> BASE
    SELF -.항상 포함.-> BASE
```